Loading and Analyzing Data
==========================

In [136]:
import pandas as pd  # programmers like shortening names for things, so they usually import pandas as "pd"
import altair as alt # again with the shortening of names
import requests # we use it for downloading the data so we don't have to save it on GitHub (too big!)
import zipfile # for de-compressing the files we download from the EPA

## Load the air quality data
The EPA published PM2.5 daily summary files annually [on their website](https://aqs.epa.gov/aqsweb/airdata/download_files.html#Daily). This data is the "PM2.5 FRM/FEM Mass (88101)" dataset.
Pandas understands what a CSV file is, so here we can just load them into two `DataFrame`s. A data frame is simply one representation of a table of data. It is the most important form of storage for working with data in pandas.

In [139]:
# Download the data from the EPA website
data_file_urls = [
    'https://aqs.epa.gov/aqsweb/airdata/daily_88101_2020.zip',
    'https://aqs.epa.gov/aqsweb/airdata/daily_88101_2019.zip'
]
# copied this example from https://stackoverflow.com/questions/16694907/download-large-file-in-python-with-requests
for url in data_file_urls:
    local_filename = "data/{}".format(url.split('/')[-1])
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        with open(local_filename, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192): 
                f.write(chunk)
# and unzip the files
files_to_unzip = ["data/{}".format(url.split('/')[-1]) for url in data_file_urls]
for f in files_to_unzip:
    with zipfile.ZipFile(f,"r") as zip_ref:
        zip_ref.extractall("data")

In [140]:
air_2019_df = pd.read_csv("data/daily_88101_2019.csv")
air_2020_df = pd.read_csv("data/daily_88101_2020.csv")
air_2020_df.head() # this helpfully prints out the first few rows with headers to preview the data

In [21]:
"{} rows for 2019, {} rows for 2020".format(air_2019_df.shape[0], air_2019_df.shape[0])

## Aggregate and average MA data by city
Let's compare MA data by city in 2020 and 2019.

In [128]:
# Step 1 - filter the data down by state name

is_MA_data = air_2020_df['State Name'] == "Massachusetts"
air_MA_2020_df = air_2020_df[is_MA_data]

is_MA_data = air_2019_df['State Name'] == "Massachusetts"
air_MA_2019_df = air_2019_df[is_MA_data]

"{} reports for MA in 2019, {} reports for MA in 2020".format(air_MA_2019_df.shape[0], air_MA_2020_df.shape[0])

In [129]:
# now trim down to just the columns we care about so it is easier to understand
interesting_columns = ['City Name', 'Latitude', 'Longitude', 'Arithmetic Mean']
air_MA_2020_df = pd.DataFrame(air_MA_2020_df, columns=interesting_columns)
air_MA_2019_df = pd.DataFrame(air_MA_2019_df, columns=interesting_columns)
air_MA_2019_df

In [132]:
# now group all the records by city and average them
avg_by_city_2020_MA_df = air_MA_2020_df.groupby('City Name').mean()\
    .reset_index()\
    .rename(columns={'City Name': 'City', 'Arithmetic Mean': 'Mean'})
avg_by_city_2019_MA_df = air_MA_2019_df.groupby('City Name').mean()\
    .reset_index()\
    .rename(columns={'City Name': 'City', 'Arithmetic Mean': 'Mean'})
# now we need to add in a year column so we can tell the data apart!
avg_by_city_2020_MA_df['year'] = 2020
avg_by_city_2019_MA_df['year'] = 2019
# now we can just contacetane the two dataframes to get all our data in one place
ma_city_avg_df = avg_by_city_2019_MA_df.append(avg_by_city_2020_MA_df)
ma_city_avg_df.to_csv('data/MA-city-year-avg.csv')
ma_city_avg_df

## Visually Inspect the Data

In [131]:
alt.Chart(ma_city_avg_df, height=300).mark_bar().encode(
    alt.X('year:N'),
    alt.Y('Mean'),
    color='year:N',
    column=alt.Column(field='City', type='ordinal', spacing=10)
).properties( 
    title="MA City Average PM2.5 (by year)",
)

In [ ]:
-- some sql